# A股智能预测 — 云端 GPU 训练
在 Google Colab 上用免费 T4 GPU 训练模型，完成后下载 checkpoint 到本地使用。

### 使用步骤
1. 点击菜单栏 `Runtime` → `Change runtime type` → 选择 `T4 GPU`
2. 依次运行每个单元格
3. 训练完成后，checkpoint 会自动下载

In [ ]:
# ═══════════════════════════════════════
# Step 1: 安装依赖 + 挂载 Google Drive
# ═══════════════════════════════════════
!pip install -q efinance pandas numpy scikit-learn torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q py-mini-racer  # efinance dependency

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_mem // 1024**2} MB')
    torch.backends.cudnn.benchmark = True

# 挂载 Google Drive 用于保存模型
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/stock_models'
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# ═══════════════════════════════════════
# Step 2: 加载模型代码（从本地 zip 或 GitHub）
# ═══════════════════════════════════════

# 如果代码已打包上传，解压即可；否则从本地上传 stock_predictor.zip
import zipfile, os, sys

ZIP_PATH = '/content/stock_predictor.zip'
if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('/content/stock_predictor')
    sys.path.insert(0, '/content/stock_predictor')
    print('代码已解压')
else:
    print('请先上传 stock_predictor.zip 到 /content/')
    print('在左侧 Files 面板点击上传按钮')

In [ ]:
# ═══════════════════════════════════════
# Step 3: 训练配置
# ═══════════════════════════════════════

# === 修改这里选择要训练的股票和模型 ===
STOCK_CODE = input('输入股票代码 (如 000001.SZ): ') or '000001.SZ'

MODEL_CHOICES = {
    '1': 'Node Transformer (图注意力)',
    '2': 'FT-iTransformer (时频协同)',
    '3': 'PINN (物理约束)',
    '4': '集成-加权平均',
    '5': '集成-多数投票',
    '6': '集成-堆叠法 (Stacking)',
    '7': '集成-PINN Guard',
    '8': '所有模型',
}

print('\n模型选择:')
for k, v in MODEL_CHOICES.items():
    print(f'  {k}. {v}')

choice = input('选择模型 (1-8, 默认8=所有模型): ') or '8'
MODEL_NAME = MODEL_CHOICES.get(choice, '所有模型')
EPOCHS = int(input('训练轮数 (默认100): ') or '100')

print(f'\n=== 训练配置 ===')
print(f'股票: {STOCK_CODE}')
print(f'模型: {MODEL_NAME}')
print(f'轮数: {EPOCHS}')
print(f'设备: {"GPU" if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ═══════════════════════════════════════
# Step 4: 获取数据 + 特征工程
# ═══════════════════════════════════════

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import efinance as ef

print(f'正在获取 {STOCK_CODE} 数据...')

# 获取最近 60 天 5 分钟数据
df = ef.stock.get_quote_history(STOCK_CODE, klt=5, beg=(datetime.now() - timedelta(days=60)).strftime('%Y%m%d'))
if df is None or df.empty:
    print('efinance 获取失败，尝试用 pandas-datareader...')
    # 备选数据源
else:
    # 统一列名
    col_map = {
        '日期': 'trade_time', '开盘': 'open', '收盘': 'close',
        '最高': 'high', '最低': 'low', '成交量': 'volume', '成交额': 'amount'
    }
    df.rename(columns={k: v for k, v in col_map.items() if k in df.columns}, inplace=True)
    if 'trade_time' in df.columns:
        df['trade_time'] = pd.to_datetime(df['trade_time'])
        df.sort_values('trade_time', inplace=True)
    print(f'获取到 {len(df)} 条数据')
    print(df.tail(3))

In [ ]:
# ═══════════════════════════════════════
# Step 5: 导入模型并训练
# ═══════════════════════════════════════

import sys
sys.path.insert(0, '/content/stock_predictor')

from model.trainer import train_model
from model.dataset import StockDataset
from data.preprocessor import preprocess, build_targets
from data.features import compute_all_indicators
from data.feature_selector import select_and_save, get_feature_mask
from config import load_config
from model.ft_transformer import FT_iTransformerWrapper
from model.node_transformer import NodeTransformerWrapper
from model.pinn_model import PINNWrapper
from model.ensemble import (
    EnsembleWeightedWrapper, EnsembleVotingWrapper,
    EnsembleStackingWrapper, EnsemblePINNGuardWrapper, AllModelsWrapper
)
from gui.workers import TrainingWorker

# 模型注册表
MODEL_REGISTRY = {
    'Node Transformer (图注意力)': NodeTransformerWrapper,
    'FT-iTransformer (时频协同)': FT_iTransformerWrapper,
    'PINN (物理约束)': PINNWrapper,
    '集成-加权平均': EnsembleWeightedWrapper,
    '集成-多数投票': EnsembleVotingWrapper,
    '集成-堆叠法 (Stacking)': EnsembleStackingWrapper,
    '集成-PINN Guard': EnsemblePINNGuardWrapper,
    '所有模型': AllModelsWrapper,
}

print(f'\n=== 开始训练 ===')
print(f'模型: {MODEL_NAME}')

# 加载配置
config = load_config()
config.model.epochs = EPOCHS
config.model.checkpoint_dir = SAVE_DIR

# 数据预处理
print('计算技术指标...')
ind = compute_all_indicators(df)
for col in ind.columns:
    if col not in df.columns:
        df[col] = ind[col].values

print('MRMR 特征选择...')
selected_features = select_and_save(df, STOCK_CODE, SAVE_DIR, k=None, horizon=10)
print(f'选定 {len(selected_features)} 个特征')

# 构建数据集
feat_arr, scaler = preprocess(df, fit_scaler=True)
mask = get_feature_mask(selected_features)
if feat_arr.shape[1] == len(mask):
    feat_arr = feat_arr[:, mask]
    from sklearn.preprocessing import StandardScaler
    sc2 = StandardScaler(); sc2.fit(feat_arr); scaler = sc2

raw_close = df['close'].values.astype(float)
direction, price_change = build_targets(raw_close, horizon=10)
targets = np.stack([direction, price_change], axis=1)
epoch_ts = pd.to_datetime(df['trade_time']).apply(lambda t: t.timestamp()).values.astype(float)

dataset = StockDataset([feat_arr], [targets], config.model.seq_len, horizon=10,
                        dense=True, timestamps=[epoch_ts])
print(f'数据集: {len(dataset)} 个样本')

# 创建模型
input_dim = len(selected_features)
model_cls = MODEL_REGISTRY[MODEL_NAME]
model = model_cls(
    input_dim=input_dim, d_model=config.model.d_model,
    lstm_hidden=config.model.lstm_hidden, lstm_layers=config.model.lstm_layers,
    transformer_layers=config.model.transformer_layers, nhead=config.model.nhead,
    dropout=config.model.dropout, max_seq_len=config.model.seq_len,
    patch_len=config.model.patch_len
)

# 使用 TrainingWorker (后台线程)
worker = TrainingWorker(
    model, dataset, config, SAVE_DIR, scaler=scaler,
    ts_code=STOCK_CODE, selected_features=selected_features,
    epochs_override=EPOCHS, model_type=MODEL_NAME
)

# 同步等待训练完成 (Colab 环境下不需要异步)
import time
worker.start()
while worker.isRunning():
    time.sleep(0.5)

print('\n训练完成!')
print(f'模型已保存到: {SAVE_DIR}/{STOCK_CODE}_best_model.pt')

In [ ]:
# ═══════════════════════════════════════
# Step 6: 打包下载模型文件
# ═══════════════════════════════════════

import shutil
import glob

# 找出所有相关文件
model_files = glob.glob(f'{SAVE_DIR}/{STOCK_CODE}*')
print(f'模型文件 ({len(model_files)} 个):')
for f in model_files:
    size_mb = os.path.getsize(f) / 1024**2
    print(f'  {os.path.basename(f)} ({size_mb:.1f} MB)')

# 打包下载
!zip -j /content/stock_model.zip {' '.join(model_files)}

from google.colab import files
files.download('/content/stock_model.zip')

print('\n下载完成后，解压到本地 D:/AI/stock_predictor/pretrained/ 目录即可使用')